# 원판 후향 테스트 · 수행자 3명, 영상 7편

동작 구간에서 미리 고른 35장으로 최종 원판 박스를 채점합니다. 동작 전체에서는 좌표가 어디에서 끊겼는지 확인합니다. 공개 CSV만으로 결과표와 그래프를 재현할 수 있습니다. 원본과 최종 박스의 FiftyOne 대조에는 로컬 영상과 가중치가 필요합니다.

In [ ]:
from pathlib import Path
import os
os.environ["FIFTYONE_DO_NOT_TRACK"] = "1"
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
from IPython.display import display
from wlcv.config import ROOT

local = ROOT / "outputs/plate_holdout/20261127_mac_cpu"
folder = local if (local / "by_video.csv").is_file() else ROOT / "docs/results/plate_holdout"
local.mkdir(parents=True, exist_ok=True)
people = pd.read_csv(folder / "by_person.csv").set_index("person_id")
videos = pd.read_csv(folder / "by_video.csv").set_index("video_id")
samples = pd.read_csv(folder / "scored_samples.csv").query("scored")
gaps = pd.read_csv(folder / "missing_ranges.csv").query("scope == 'motion'")

## 선수별 결과표
수동 박스와 최종 박스의 IoU가 0.50 이상이면 정탐입니다. 좌표가 남은 동작 프레임 수는 정답 박스가 없는 장면까지 포함합니다.

In [ ]:
table = people[["scored", "tp", "fn", "fp", "motion_frames", "motion_missing", "motion_longest"]].copy()
table.insert(0, "videos", videos.groupby("person_id").size())
table.loc["전체"] = table.sum()
table.loc["전체", "motion_longest"] = people.motion_longest.max()
table.insert(5, "recall", (100 * table.tp / (table.tp + table.fn)).round(1))
table.columns = ["영상 수", "판독 장수", "정탐", "미탐", "오탐", "재현율 (%)", "동작 프레임", "좌표 누락", "최장 연속 누락"]
table.index.name = "선수"
table.to_csv(local / "test_results_ko.csv", encoding="utf-8-sig")
display(table)

## 원판이 맞았는지와 좌표가 이어지는지를 나란히 보기
왼쪽 숫자는 프레임별 수동 박스와의 IoU입니다. 최종 박스가 없으면 ‘미탐’으로 표시합니다. 오른쪽은 각 영상의 동작 시작을 0으로 맞춰 표시합니다.

In [ ]:
for family in ["Apple SD Gothic Neo", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:
    if family in font_manager.get_font_names():
        plt.rcParams["font.family"] = family
        break
plt.rcParams["axes.unicode_minus"] = False
fig, (boxes, tracks) = plt.subplots(1, 2, figsize=(10.8, 4.8), sharey=True,
                                   gridspec_kw={"width_ratios": [1, 1.25]})
green, red = "#13675d", "#ae3526"
labels = []
for y, (video, row) in enumerate(videos.iterrows()):
    athlete, _, lift, take = video.split("_")
    labels.append(f"{athlete}  {'하프' if lift == 'h' else '풀'} {int(take)}")
    group = samples[samples.video_id == video].sort_values("frame")
    for x, point in enumerate(group.itertuples()):
        boxes.text(x, y, f"{point.iou:.2f}" if point.tp else "미탐", ha="center", va="center",
                   color=green if point.tp else red, weight="bold" if point.fn else "normal",
                   bbox=dict(boxstyle="round,pad=0.32", linewidth=0,
                             facecolor="#e8f3ee" if point.tp else "#fce2db"))
    # 모든 동작 프레임의 좌표 유무를 표시합니다. 누락 시점은 끊고 X로 표시합니다.
    present = np.ones(row.motion_frames, dtype=bool)
    for gap in gaps[gaps.video_id == video].itertuples():
        present[gap.start-row.start:gap.end-row.start+1] = False
    x = np.arange(row.motion_frames)
    tracks.plot(x[present], np.full(present.sum(), y), "|", ms=8, color=green)
    tracks.scatter(x[~present], np.full((~present).sum(), y), marker="x", color=red, s=58, zorder=3)
    if not present.all():
        tracks.text(row.motion_frames+2, y, "원본 54번 · 1프레임 누락", va="center", color=red, fontsize=9)
boxes.set(xticks=range(5), xticklabels=["시작", "25%", "50%", "75%", "끝"], xlim=(-.5,4.5),
          yticks=range(len(labels)), yticklabels=labels, xlabel="동작 구간에서 판독한 위치")
boxes.set_ylim(len(labels)-.5, -.5)
tracks.set(xlabel="동작 시작 이후 프레임", xlim=(-3, max(videos.motion_frames)+3))
boxes.set_title("수동으로 확인한 35장 · 박스 IoU", loc="left", weight="bold", pad=15)
tracks.set_title("동작 434프레임 · 최종 좌표 유무", loc="left", weight="bold", pad=15)
for ax in (boxes, tracks):
    ax.spines[["top","right","left"]].set_visible(False)
    ax.tick_params(length=0, pad=6)
fig.tight_layout(w_pad=2.5)
fig.savefig(local / "test_overview.png", dpi=160, bbox_inches="tight", facecolor="white")
plt.show()

## FiftyOne 검수
‘scored’ 뷰는 35장입니다. ‘final-misses’은 실제 배포 출력으로 평가합니다. ‘a07-54-context’에서 53번과 55번은 문맥 확인용으로만 봅니다. `ground_truth`는 수동 박스, `deployment`는 사용자가 받는 최종 박스입니다. 54번에서 `baseline_raw`, `cutmix_raw`를 하나씩 켜면 수행자 필터 전 원판을 볼 수 있습니다. `people_candidates`는 자세 모델의 사람 후보입니다.

첫 검수 셀에서 라이브러리와 로컬 데이터셋을 불러옵니다. 저장 뷰의 장수가 출력될 때까지 기다려 주세요. 로딩 도중 중단한 뒤 `__reduce_cython__` 오류가 반복되면 위쪽 **재시작 → 모두 실행**을 선택하세요. 마지막 셀은 브라우저에서 열 수 있는 로컬 검수 링크를 표시합니다. 검수 중에는 커널을 실행한 상태로 두세요.

In [ ]:
dataset = None
if (local / "runs.csv").is_file():
    print("FiftyOne 검수 자료를 불러오는 중입니다…", flush=True)
    from wlcv.evaluation.plate_holdout_review import review_dataset

    dataset = review_dataset(local)
    for name in dataset.list_saved_views():
        print(name, len(dataset.load_saved_view(name)), "장")
else:
    print("FiftyOne 원본 대조에는 로컬 영상·가중치와 영상별 실행 기록(runs.csv)이 필요합니다.")

In [ ]:
if dataset is not None:
    import fiftyone as fo
    from IPython.display import Markdown, display
    from urllib.parse import quote

    view_name = "a07-54-context"
    session = fo.launch_app(view=dataset.load_saved_view(view_name), address="127.0.0.1", port=5178, auto=False)
    url = f"{session.url.rstrip('/')}/datasets/{quote(dataset.name)}?view={quote(view_name)}"
    display(Markdown(f"[FiftyOne에서 53·54·55번 프레임 열기]({url})"))

## 해석

판독한 35장 중 34장의 최종 박스가 수동 정답과 맞았습니다. 54번은 두 원판 모델 모두 정답 박스를 찾았으나 수행자 선택이 보류됐습니다. 53번과 55번 좌표는 직접 검출됐고, 54번을 지나는 한 프레임 단위 속도는 계산할 수 없습니다. 원판이 없는 장면에서의 오탐률과 허브 위치 오차는 이 자료로 평가할 수 없습니다. 이 후향 테스트 결과를 바탕으로 배포 설정을 조정하지 않습니다.